# Paper Hunter quickstart

Download the PDFs for a list of DOIs in a few lines of Python. For each DOI, Paper Hunter tries Unpaywall → OpenAlex → Crossref → Sci-Hub and stops at the first PDF it finds.

Prefer clicking to coding? Double-click the starter for your system in the project folder (`start_windows.bat`, `start_macos.command` or `start_linux.sh`) to open the same thing as a web page.

In [ ]:
# 1st line installs the dependencies; 2nd line always fetches the latest Paper Hunter code from GitHub
# (pip skips a reinstall when the version number is unchanged).
# Restart the kernel after running this cell, otherwise the previously imported code keeps running.
%pip install -q "paper-hunter @ git+https://github.com/rasoulnorouzi/paper_hunter.git"
%pip install -q --force-reinstall --no-deps "paper-hunter @ git+https://github.com/rasoulnorouzi/paper_hunter.git"

In [ ]:
import logging
from pathlib import Path

from paper_hunter import load_dois_from_csv, run_bulk_download, zip_downloads, zip_downloads_split

# Progress counters are always printed. Uncomment to also see the detailed per-source logs
# (Google Colab hides them by default):
# logging.getLogger().setLevel(logging.INFO)

In [ ]:
# The DOI column is detected automatically; empty rows and duplicates are skipped.
# Uses the local sample_doi.csv when present (e.g. in a clone), otherwise the copy on GitHub.
doi_file = 'sample_doi.csv'
if not Path(doi_file).exists():
    doi_file = 'https://raw.githubusercontent.com/rasoulnorouzi/paper_hunter/main/examples/sample_doi.csv'
dois = load_dois_from_csv(doi_file)
len(dois)

In [ ]:
# Columns: doi, success, source (which site delivered the PDF), file
results_df = run_bulk_download(dois, download_dir='fulldownloads')
results_df.head()

In [ ]:
success_count = int(results_df['success'].sum())
total = len(results_df)
print(f'Success: {success_count}/{total} ({success_count / total:.1%})')
print('Not found:', results_df.loc[~results_df['success'], 'doi'].tolist())
print('Summary CSV: fulldownloads/download_summary.csv')

## Zip the results

One ZIP with every PDF and the summary CSV, or split into several ZIPs (each with its own summary).

In [ ]:
zip_downloads('fulldownloads', zip_name='papers')                  # papers.zip
zip_downloads_split('fulldownloads', n_splits=3, zip_name='papers')  # papers_part1_of_3.zip ... part3

## Adding a source

Create a subclass of `PDFDownloader` in `paper_hunter/sources.py` that implements `try_download(doi)`, then add it to the list in `make_manager` in `paper_hunter/core.py`. The notebook code stays the same.